# δ-Mem Compound-Stack Test Matrix

Cross-base retrofit study — Qwen3-4B-Instruct-2507 (primary) + SmolLM3-3B (secondary, adapter trained in pre-cell T1).

Tests δ-Mem, sliding-window attention, StreamingLLM, speculative decoding, and native MTP as compound levers over a vanilla full-attention baseline.

**Run on Kaggle:** `File → Import Notebook → URL` with the GitHub raw URL.  
**Stop a run:** push `control/STATUS=stop` to the repo from anywhere; the next cell boundary will exit gracefully.  
**Auto-stop:** wall-clock budget (default 8 h) or natural completion.

> Full spec in [`docs/spec.md`](../docs/spec.md). Scaffold version — cell dispatch is stubbed; runs report `status: scaffold-stub`.

## 1. Environment detection and repo sync

In [ ]:
import os, sys, subprocess, pathlib, shutil, json, time, textwrap

REPO_URL = "https://github.com/jamesburton/delta-mem-smollm3-3b.git"
REPO_NAME = "delta-mem-smollm3-3b"

def detect_env():
    if pathlib.Path("/kaggle").exists():
        return "kaggle"
    if pathlib.Path("/content").exists():
        return "colab"
    return "local"

ENV = detect_env()
print("environment:", ENV)

if ENV == "kaggle":
    WORKDIR = pathlib.Path("/kaggle/working") / REPO_NAME
elif ENV == "colab":
    WORKDIR = pathlib.Path("/content") / REPO_NAME
else:
    # local — assume we're already in the repo
    WORKDIR = pathlib.Path.cwd()
    if WORKDIR.name != REPO_NAME and not (WORKDIR / "harness").exists():
        raise SystemExit("Run this notebook from the repo root locally, or on Kaggle/Colab.")

if ENV in ("kaggle", "colab") and not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
elif ENV in ("kaggle", "colab"):
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--rebase"], check=True)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("workdir:", WORKDIR)

## 2. Install dependencies

In [ ]:
if ENV in ("kaggle", "colab"):
    subprocess.run(["bash", "scripts/kaggle_bootstrap.sh"], check=True)
else:
    print("Local run — assuming requirements already installed.")

## 3. Verify GPU and imports

In [ ]:
import torch, transformers, accelerate
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB  CC {p.major}.{p.minor}")
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

from harness import state, control, cells, oracle
print("harness version:", __import__('harness').__version__)

## 4. Oracle authentication

Auto-detects: `claude` CLI (subscription) → `ANTHROPIC_API_KEY` (Python SDK) → unavailable.

On Kaggle, the cleanest path is to either:
1. install Claude Code CLI in this kernel and run `!claude login` interactively (device-flow), **or**
2. set `ANTHROPIC_API_KEY` as a Kaggle secret (Add-ons → Secrets) and the harness will pick it up automatically.

If neither, runs proceed without the frontier-ceiling row in the summary.

### 4a. (Optional) Install Claude CLI and sign in to your subscription

Skip this section if you don't need oracle / LLM-as-judge scoring, or if you have set `ANTHROPIC_API_KEY` as a Kaggle secret and prefer the API-credits path.

Run the next three cells in order:
1. **Install** Claude Code CLI into the kernel.
2. **Restore credentials from a Kaggle secret** (if you've set `CLAUDE_CREDENTIALS_B64`) — this skips the interactive login entirely.
3. **Interactive `claude login`** (device-flow OAuth) — only run if cell 2 reported the secret wasn't found.

All three cells are idempotent — safe to re-run.

#### Three ways to authenticate, in increasing setup-effort order

**(a) Interactive `claude login` in the cell below.** Cheapest. The first-run theme picker is auto-skipped via a pre-seeded `~/.claude/settings.json`. The device-flow URL prints to the cell output; you complete sign-in on another device.

**(b) Use Kaggle's Console pane.** If the interactive cell still misbehaves on your kernel (Jupyter's `!` shell isn't a real PTY), open the **Console** icon in Kaggle's right-side panel — that's a proper terminal where `claude login` works normally with arrow keys. Auth state persists into the notebook kernel.

**(c) Transplant credentials from a local login.** Do `claude login` once on a machine with a real terminal (your laptop), grab `~/.claude/.credentials.json` (and optionally `settings.json`), base64-encode it, and add as a Kaggle secret named `CLAUDE_CREDENTIALS_B64`. The cell below this markdown auto-restores when the secret is present. No interactive auth needed — until the OAuth token expires.

To encode the credentials on **macOS / Linux**:

```bash
base64 -w0 ~/.claude/.credentials.json
```

On **Windows PowerShell**:

```powershell
[Convert]::ToBase64String([IO.File]::ReadAllBytes("$env:USERPROFILE\.claude\.credentials.json"))
```

Paste the resulting string into a Kaggle secret labelled exactly `CLAUDE_CREDENTIALS_B64` and attach it to this notebook.

In [ ]:
# Install Claude Code CLI (idempotent — checks if already on PATH).
import shutil, json, pathlib

if shutil.which("claude") is None:
    # Official installer puts the binary under ~/.local/bin
    subprocess.run(
        "curl -fsSL https://claude.ai/install.sh | bash",
        shell=True, check=True,
    )
    local_bin = os.path.expanduser("~/.local/bin")
    if local_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{local_bin}:{os.environ['PATH']}"
else:
    print("claude CLI already on PATH:", shutil.which("claude"))

# Pre-seed ~/.claude/settings.json with a theme so the interactive first-run
# theme picker doesn't block in Jupyter (the cell isn't a real PTY, so arrow
# keys / number selection don't reach the TUI).
config_dir = pathlib.Path.home() / ".claude"
config_dir.mkdir(parents=True, exist_ok=True)
settings_path = config_dir / "settings.json"
if not settings_path.exists():
    settings_path.write_text(json.dumps({"theme": "dark"}))
    print(f"seeded {settings_path} with theme=dark (skips first-run picker)")
else:
    print(f"{settings_path} already exists; not overwriting")

# Verify
v = subprocess.run(["claude", "--version"], capture_output=True, text=True)
print(v.stdout.strip() or v.stderr.strip())

In [ ]:
# Optional: restore Claude credentials from a Kaggle secret.
#
# If you set `CLAUDE_CREDENTIALS_B64` (and optionally `CLAUDE_SETTINGS_B64`)
# in Kaggle Add-ons → Secrets, this cell writes them to ~/.claude/ and the
# interactive `claude login` cell below becomes unnecessary.
#
# If the secret isn't set, this cell prints a note and does nothing — the
# interactive login cell below is the fallback.

import base64
import pathlib

CREDS_SECRET = "CLAUDE_CREDENTIALS_B64"
SETTINGS_SECRET = "CLAUDE_SETTINGS_B64"

restored = False

if ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()

        # Credentials (required for skip-login)
        try:
            blob = us.get_secret(CREDS_SECRET)
        except Exception:
            blob = None

        if blob:
            raw = base64.b64decode(blob)
            creds_path = pathlib.Path.home() / ".claude" / ".credentials.json"
            creds_path.parent.mkdir(parents=True, exist_ok=True)
            creds_path.write_bytes(raw)
            try:
                creds_path.chmod(0o600)
            except Exception:
                pass
            print(f"restored credentials → {creds_path}")
            restored = True

            # Optional: also restore settings (theme, prefs) if available
            try:
                sblob = us.get_secret(SETTINGS_SECRET)
            except Exception:
                sblob = None
            if sblob:
                sraw = base64.b64decode(sblob)
                settings_path = pathlib.Path.home() / ".claude" / "settings.json"
                settings_path.write_bytes(sraw)
                print(f"restored settings → {settings_path}")
        else:
            print(f"no {CREDS_SECRET} secret on this notebook; "
                  f"use the interactive login cell below (or skip if you don't need the oracle)")
    except ImportError:
        # kaggle_secrets unavailable — we're not on Kaggle, or it's a sandboxed kernel
        print("kaggle_secrets unavailable; skipping credential restore")
else:
    print(f"env={ENV}; credential restore only runs on Kaggle")

if restored:
    # Quick check — `claude --version` should still work, and if the token is
    # valid, downstream calls via `claude --print` or the harness oracle will too.
    v = subprocess.run(["claude", "--version"], capture_output=True, text=True)
    print("claude:", v.stdout.strip() or v.stderr.strip())
    print("✓ credentials restored — you can SKIP the `claude login` cell below.")

In [ ]:
# Device-flow login. SKIP this cell if the cell above said
# "✓ credentials restored" — you're already authed.
#
# Otherwise: this will print a URL. Open it in any browser, sign in with
# your Claude.ai Pro/Max subscription, paste the displayed code back when
# prompted in the cell output below.
#
# If it hangs waiting for input, see the markdown above option (b):
# cancel this cell and run `claude login` from a Kaggle Console pane
# (right-side panel → Console).

if restored:
    print("credentials already restored from secret — skipping interactive login")
else:
    get_ipython().system("claude login")

In [ ]:
# Optionally pull ANTHROPIC_API_KEY from Kaggle secrets into env
if ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        try:
            os.environ["ANTHROPIC_API_KEY"] = us.get_secret("ANTHROPIC_API_KEY")
            print("loaded ANTHROPIC_API_KEY from Kaggle secrets")
        except Exception as e:
            print("no ANTHROPIC_API_KEY secret:", e)
    except ImportError:
        pass

auth_mode = oracle.detect_auth()
print("oracle auth mode:", auth_mode)
if auth_mode == "unavailable":
    print("⚠️  Runs will record `oracle: unavailable` and skip the ceiling row.")

## 5. Git push auth (for committing per-cell results)

Kaggle: set `GH_PAT_DELTA_MEM_TESTS` as a secret (a PAT with `repo` scope on this repository only). The notebook configures git to use it for pushes.

Local: assumes you already have `gh auth` or a pushable remote configured.

In [ ]:
if ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        pat = UserSecretsClient().get_secret("GH_PAT_DELTA_MEM_TESTS")
        auth_url = REPO_URL.replace("https://", f"https://x-access-token:{pat}@")
        subprocess.run(["git", "remote", "set-url", "origin", auth_url], check=True)
        subprocess.run(["git", "config", "user.email", "kaggle-bot@local"], check=True)
        subprocess.run(["git", "config", "user.name", "kaggle-bot"], check=True)
        print("git push auth configured")
    except Exception as e:
        print("⚠️  no GH_PAT_DELTA_MEM_TESTS secret — checkpoints will NOT be pushed:", e)
else:
    print("local run — assuming existing git remote auth")

## 6. Load or create run state

In [ ]:
import datetime

STAGE = os.environ.get("STAGE", "S3")    # set explicitly if running on S1/S2 locally
WALL_CLOCK_BUDGET = int(os.environ.get("WALL_CLOCK_BUDGET", 8 * 3600))

run_id = f"{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}-{ENV}-{STAGE}"
run_state = state.load_or_create(run_id=run_id, stage=STAGE, budget_seconds=WALL_CLOCK_BUDGET)

print("run_id:", run_state.run_id)
print("stage:", run_state.stage)
print("completed cells so far:", run_state.completed_cells)
print(f"wall-clock budget: {run_state.wall_clock_budget_seconds}s ({run_state.wall_clock_budget_seconds/3600:.1f}h)")

## 7. Dispatch loop

For each pending cell in stage order:
1. Check `control/STATUS` and wall-clock — exit gracefully if either trips.
2. Run the cell (stubbed in this scaffold — real runners land in the implementation phase).
3. Write `results/{cell.id}.json`.
4. Commit + push the result back to the repo.
5. Update `state.json` and push it too.

In [ ]:
from harness.runners import hf_runner
from harness.runners.hf_runner import RunConfig

def commit_and_push(paths, message):
    subprocess.run(["git", "add", *paths], check=True)
    r = subprocess.run(["git", "commit", "-m", message], capture_output=True, text=True)
    if r.returncode != 0 and "nothing to commit" not in r.stdout + r.stderr:
        raise RuntimeError(f"commit failed: {r.stderr}")
    push = subprocess.run(["git", "push"], capture_output=True, text=True)
    if push.returncode != 0:
        print("  ⚠️ push failed (continuing):", push.stderr.strip())

def _utc_now_iso():
    """Timezone-aware UTC ISO 8601 with 'Z' suffix (utcnow() deprecated in 3.12+)."""
    return datetime.datetime.now(datetime.UTC).isoformat().replace("+00:00", "Z")

# Phase 1 covers cells 1, 2, 6, 7. All others write status=phase-2-plus.
PHASE_1 = {"1", "2", "6", "7"}

base_cfg = RunConfig(
    target_tokens=int(os.environ.get("NIH_TARGET_TOKENS", "4000")),
    n_needles=3,
    max_new_tokens=int(os.environ.get("MAX_NEW_TOKENS", "256")),
    seed=0,
    dtype=os.environ.get("DTYPE", "bfloat16"),
    device="cuda" if torch.cuda.is_available() else "cpu",
    results_dir=WORKDIR / "results" / STAGE,
)

results_dir = WORKDIR / "results" / STAGE
results_dir.mkdir(parents=True, exist_ok=True)

for cell in cells.cells_for_stage(STAGE):
    if cell.id in run_state.completed_cells:
        print(f"✓ skipping {cell.id} ({cell.title}) — already done")
        continue
    if cell.blocked_by and cell.blocked_by not in run_state.completed_cells:
        print(f"⏸ skipping {cell.id} — blocked by {cell.blocked_by}")
        continue

    stop, reason = control.should_stop(run_state)
    if stop:
        print(f"🛑 stopping early: {reason}")
        break

    run_state.current_cell = cell.id
    run_state.save()
    print(f"▶ running cell {cell.id}: {cell.title}")

    if cell.id in PHASE_1 and cell.stack == "hf":
        try:
            result = hf_runner.run(cell, base_cfg)
        except Exception as e:
            result = {
                "cell_id": cell.id, "title": cell.title,
                "status": "failed", "error": repr(e),
                "timestamp": _utc_now_iso(),
            }
    else:
        result = {
            "cell_id": cell.id, "title": cell.title,
            "status": "phase-2-plus",
            "note": "Not implemented in Phase 1. See docs/spec.md.",
            "timestamp": _utc_now_iso(),
        }

    out_path = results_dir / f"cell-{cell.id}.json"
    out_path.write_text(json.dumps(result, indent=2))

    run_state.mark_completed(cell.id)
    commit_and_push(
        [str(out_path.relative_to(WORKDIR)), "state.json"],
        f"cell {cell.id}: {cell.title} [{run_state.stage}/{ENV}] -> {result['status']}",
    )
    print(f"  → wrote {out_path.relative_to(WORKDIR)} ({result['status']}) and pushed")

## 8. Render inline summary

In [ ]:
from IPython.display import Markdown, display
from harness import summary as _summary

md = _summary.render(WORKDIR / "results", stage=STAGE)
display(Markdown(md))
(WORKDIR / "results" / "summary.md").write_text(md)
commit_and_push(["results/summary.md"], f"summary [{STAGE}/{ENV}]")

## 9. Self-terminate

Final commit pushed. After this cell, **also stop the Kaggle session manually** (top-right ⋯ → Stop session) so the kernel doesn't sit idle counting against your quota.

`os._exit(0)` short-circuits any later cells if someone hits Run All twice.

In [ ]:
print("All done. Stop the Kaggle session from the UI to release the GPU.")
# Uncomment for true self-terminate (kills the kernel):
# import os; os._exit(0)